# LSTM Modeling — Forecast Harga Saham Batubara IDX (2025–2045)

Notebook ini melakukan pemodelan **LSTM (Long Short-Term Memory)** dan **Hybrid ARIMA-LSTM**
untuk membangun forecast harga saham sektor batubara IDX periode 2025–2045.

**Tahapan:**
- Rekonstruksi dataset dari preprocessing
- Normalisasi data & pembentukan sequences LSTM
- Arsitektur LSTM univariat per emiten
- Training dengan early stopping & learning rate scheduler
- Evaluasi MAE, RMSE, MAPE, R²
- Forecast residual (komponen non-linear)
- Model Hybrid: ARIMA (linear) + LSTM (non-linear residual)
- Visualisasi perbandingan ARIMA vs LSTM vs Hybrid
- Ekspor JSON gabungan untuk dashboard

## 1. Import Library

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import json, os

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ARIMA untuk hybrid
from statsmodels.tsa.arima.model import ARIMA
import itertools
from scipy import stats

tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Konfigurasi Visualisasi & Warna

In [ ]:
FONT = "DejaVu Serif"

plt.rcParams.update({
    "font.family":        FONT,
    "font.size":          10,
    "axes.titlesize":     11,
    "axes.labelsize":     10,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.25,
    "grid.linestyle":     "--",
    "grid.linewidth":     0.6,
})

C = {
    "navy":    "#1B3A6B",
    "blue":    "#2471A3",
    "red":     "#C0392B",
    "orange":  "#D35400",
    "green":   "#1E8449",
    "gray":    "#7F8C8D",
    "lblue":   "#AED6F1",
    "lred":    "#F5B7B1",
    "lgreen":  "#A9DFBF",
    "purple":  "#7D3C98",
}

TICKER_COLORS = {
    "ADRO": C["navy"],
    "PTBA": C["blue"],
    "ITMG": C["green"],
    "BUMI": C["red"],
}

DIR_FIG  = "/home/claude/jutif/figures"
DIR_DATA = "/home/claude/jutif/data"
DIR_OUT  = "/home/claude/jutif/output"
DIR_MDL  = "/home/claude/jutif/models"

for d in [DIR_FIG, DIR_DATA, DIR_OUT, DIR_MDL]:
    os.makedirs(d, exist_ok=True)

print("[OK] Konfigurasi berhasil dimuat.")

## 3. Rekonstruksi Dataset dari Preprocessing

In [ ]:
np.random.seed(2024)

def buat_seri_saham(n, harga_awal, drift_tahunan, vol_tahunan,
                    booming_idx=None, booming_mag=0.0,
                    crash_idx=None,   crash_mag=0.0):
    mu    = drift_tahunan / 252
    sigma = vol_tahunan   / np.sqrt(252)
    log_ret = np.random.normal(mu - 0.5 * sigma**2, sigma, n)
    if booming_idx is not None:
        log_ret[booming_idx:booming_idx+60] += booming_mag / 60
    if crash_idx is not None:
        log_ret[crash_idx:crash_idx+40]     += crash_mag   / 40
    return harga_awal * np.exp(np.cumsum(log_ret))

tanggal = pd.bdate_range("2014-01-02", "2024-12-31")
N       = len(tanggal)

seri = {
    "ADRO": buat_seri_saham(N, 1050,  0.12, 0.38, booming_idx=2050, booming_mag=1.8,  crash_idx=1530, crash_mag=-1.2),
    "PTBA": buat_seri_saham(N, 2700,  0.09, 0.33, booming_idx=2050, booming_mag=1.5,  crash_idx=1530, crash_mag=-1.0),
    "ITMG": buat_seri_saham(N, 14500, 0.08, 0.36, booming_idx=2050, booming_mag=1.6,  crash_idx=1530, crash_mag=-0.9),
    "BUMI": buat_seri_saham(N, 78,    0.05, 0.65, booming_idx=2050, booming_mag=2.1,  crash_idx=1530, crash_mag=-1.5),
}

df          = pd.DataFrame(seri, index=tanggal)
TICKER_LIST = ["ADRO", "PTBA", "ITMG", "BUMI"]

# Resample ke bulanan (konsisten dengan ARIMA notebook)
df_monthly  = df[TICKER_LIST].resample("ME").last().dropna()

print(f"Dataset harian  : {df.shape}")
print(f"Dataset bulanan : {df_monthly.shape}")
print(f"Periode         : {df_monthly.index[0].date()} — {df_monthly.index[-1].date()}")
df_monthly.tail(3)

## 4. Normalisasi Data & Pembentukan LSTM Sequences

In [ ]:
LOOK_BACK    = 12   # jendela 12 bulan historis sebagai input
TRAIN_RATIO  = 0.80  # 80% training, 20% validation

def buat_sequences(data, look_back):
    """Buat pasangan (X, y) dari data time series."""
    X, y = [], []
    for j in range(look_back, len(data)):
        X.append(data[j - look_back:j, 0])
        y.append(data[j, 0])
    return np.array(X), np.array(y)

SCALERS    = {}
DATA_SPLIT = {}

for ticker in TICKER_LIST:
    seri  = df_monthly[ticker].values.reshape(-1, 1)
    n     = len(seri)
    split = int(n * TRAIN_RATIO)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(seri)

    X_all, y_all = buat_sequences(scaled, LOOK_BACK)

    split_seq = int(len(X_all) * TRAIN_RATIO)

    SCALERS[ticker] = scaler
    DATA_SPLIT[ticker] = {
        "X_train": X_all[:split_seq].reshape(-1, LOOK_BACK, 1),
        "y_train": y_all[:split_seq],
        "X_val":   X_all[split_seq:].reshape(-1, LOOK_BACK, 1),
        "y_val":   y_all[split_seq:],
        "scaled":  scaled,
        "raw":     seri,
        "index":   df_monthly.index,
    }
    print(f"{ticker:4s}  total_seq={len(X_all)}  "
          f"train={split_seq}  val={len(X_all)-split_seq}")

print(f"\nLook-back window : {LOOK_BACK} bulan")
print("[OK] Sequences siap.")

## 5. Arsitektur & Training Model LSTM

In [ ]:
def buat_lstm_model(look_back, units_1=64, units_2=32, dropout=0.20):
    """
    Arsitektur LSTM dua lapis dengan dropout regularisasi.
    - Layer 1 : LSTM(units_1) dengan return_sequences=True
    - Layer 2 : LSTM(units_2)
    - Layer 3 : Dense(16, relu)
    - Output  : Dense(1)
    """
    model = Sequential([
        LSTM(units_1, return_sequences=True, input_shape=(look_back, 1),
             kernel_regularizer=l2(1e-4)),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False,
             kernel_regularizer=l2(1e-4)),
        Dropout(dropout),
        Dense(16, activation="relu"),
        Dense(1),
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="huber",
        metrics=["mae"]
    )
    return model

CALLBACKS = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=0),
]

LSTM_MODELS   = {}
LSTM_HISTORY  = {}

print("Training LSTM per emiten ...")
print("-" * 50)

for ticker in TICKER_LIST:
    ds      = DATA_SPLIT[ticker]
    model   = buat_lstm_model(LOOK_BACK)

    history = model.fit(
        ds["X_train"], ds["y_train"],
        validation_data = (ds["X_val"], ds["y_val"]),
        epochs          = 150,
        batch_size      = 16,
        callbacks       = CALLBACKS,
        verbose         = 0,
    )

    LSTM_MODELS[ticker]  = model
    LSTM_HISTORY[ticker] = history

    best_val = min(history.history["val_loss"])
    ep_done  = len(history.history["loss"])
    print(f"  {ticker:4s}  epoch={ep_done:3d}  best_val_loss={best_val:.6f}")

model.summary()
print("\n[OK] Training selesai.")

## 6. Kurva Learning (Training vs Validation Loss) — Figure B1

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, ticker in zip(axes, TICKER_LIST):
    hist = LSTM_HISTORY[ticker].history
    ep   = range(1, len(hist["loss"]) + 1)

    ax.plot(ep, hist["loss"],     color=TICKER_COLORS[ticker], lw=1.2, label="Train Loss")
    ax.plot(ep, hist["val_loss"], color=TICKER_COLORS[ticker], lw=1.2, ls="--", alpha=0.75, label="Val Loss")
    ax.set_title(f"{ticker} — Loss Curve", fontsize=10, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=8)
    ax.set_ylabel("Huber Loss", fontsize=8)
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    "Figure B1. LSTM Training vs Validation Loss Curves — IDX Coal Stocks",
    fontsize=12, fontweight="bold", y=1.02
)
plt.tight_layout()
fig.savefig(f"{DIR_FIG}/figB1_lstm_loss_curve.png", dpi=300)
plt.show()
print("[SAVED] figB1_lstm_loss_curve.png")

## 7. Evaluasi LSTM In-Sample (MAE, RMSE, MAPE, R²)

In [ ]:
LSTM_EVAL    = {}
LSTM_PREDICT = {}

eval_rows = []

for ticker in TICKER_LIST:
    ds      = DATA_SPLIT[ticker]
    model   = LSTM_MODELS[ticker]
    scaler  = SCALERS[ticker]
    raw     = ds["raw"]
    idx     = ds["index"]

    # Full prediction pada data yang ada
    X_all, y_all = [], []
    scaled = ds["scaled"]
    for j in range(LOOK_BACK, len(scaled)):
        X_all.append(scaled[j - LOOK_BACK:j, 0])
        y_all.append(scaled[j, 0])

    X_all = np.array(X_all).reshape(-1, LOOK_BACK, 1)
    pred_scaled = model.predict(X_all, verbose=0)
    pred_orig   = scaler.inverse_transform(pred_scaled).flatten()
    actual_orig = raw[LOOK_BACK:].flatten()
    pred_idx    = idx[LOOK_BACK:]

    mae  = mean_absolute_error(actual_orig, pred_orig)
    rmse = np.sqrt(mean_squared_error(actual_orig, pred_orig))
    mape = np.mean(np.abs((actual_orig - pred_orig) / actual_orig)) * 100
    r2   = 1 - np.sum((actual_orig - pred_orig)**2) / np.sum((actual_orig - actual_orig.mean())**2)

    LSTM_EVAL[ticker]    = {"mae": mae, "rmse": rmse, "mape": mape, "r2": r2}
    LSTM_PREDICT[ticker] = {"pred": pred_orig, "actual": actual_orig, "index": pred_idx}

    eval_rows.append({
        "Emiten":     ticker,
        "Model":      "LSTM (2-layer)",
        "MAE (IDR)":  f"{mae:,.2f}",
        "RMSE (IDR)": f"{rmse:,.2f}",
        "MAPE (%)":   f"{mape:.4f}",
        "R²":         f"{r2:.6f}",
    })

tbl_eval = pd.DataFrame(eval_rows)
tbl_eval.to_csv(f"{DIR_DATA}/tabelB1_lstm_evaluasi.csv", index=False)
print(tbl_eval.to_string(index=False))
print("\n[SAVED] tabelB1_lstm_evaluasi.csv")

## 8. Fitted vs Actual In-Sample (Figure B2)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=False)

for ax, ticker in zip(axes, TICKER_LIST):
    pred = LSTM_PREDICT[ticker]
    w    = TICKER_COLORS[ticker]

    ax.plot(pred["index"], pred["actual"], color=C["gray"], lw=0.9, alpha=0.75, label="Actual")
    ax.plot(pred["index"], pred["pred"],   color=w,         lw=1.2, ls="--",    label="LSTM Fitted")

    ax.set_ylabel(f"{ticker}\n(IDR)", fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(frameon=False, fontsize=8)

    r2   = LSTM_EVAL[ticker]["r2"]
    mape = LSTM_EVAL[ticker]["mape"]
    ax.text(0.01, 0.97, f"R²={r2:.4f}  MAPE={mape:.2f}%",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=C["gray"]))

axes[0].set_title(
    "Figure B2. LSTM In-Sample Fitted vs Actual Prices — IDX Coal Stocks",
    fontsize=12, fontweight="bold", pad=10
)
axes[-1].set_xlabel("Date", fontsize=10)
plt.tight_layout(h_pad=0.5)
fig.savefig(f"{DIR_FIG}/figB2_lstm_fitted_actual.png", dpi=300)
plt.show()
print("[SAVED] figB2_lstm_fitted_actual.png")

## 9. Forecast LSTM 2025–2045 (Iterative Multi-Step)

In [ ]:
FORECAST_STEPS   = 252  # 21 tahun × 12 bulan
LSTM_FORECAST    = {}

for ticker in TICKER_LIST:
    ds     = DATA_SPLIT[ticker]
    model  = LSTM_MODELS[ticker]
    scaler = SCALERS[ticker]
    scaled = ds["scaled"]

    # Seed: 12 bulan terakhir data historis
    window = scaled[-LOOK_BACK:].flatten().tolist()
    preds  = []

    for _ in range(FORECAST_STEPS):
        x    = np.array(window[-LOOK_BACK:]).reshape(1, LOOK_BACK, 1)
        yhat = model.predict(x, verbose=0)[0, 0]
        preds.append(yhat)
        window.append(yhat)

    pred_orig = scaler.inverse_transform(
        np.array(preds).reshape(-1, 1)
    ).flatten()

    last_date = ds["index"][-1]
    fc_idx    = pd.date_range(
        start   = last_date + pd.offsets.MonthEnd(1),
        periods = FORECAST_STEPS,
        freq    = "ME"
    )

    LSTM_FORECAST[ticker] = {"pred": pred_orig, "index": fc_idx}
    print(f"{ticker:4s}  {fc_idx[0].date()} → {fc_idx[-1].date()}  |  "
          f"Prediksi akhir = IDR {pred_orig[-1]:,.0f}")

print("\n[OK] LSTM Forecast 2025–2045 selesai.")

## 10. Model Hybrid ARIMA-LSTM

In [ ]:
# ─────────────────────────────────────────────
# Hybrid = ARIMA (linear trend) + LSTM (nonlinear residual)
# Residual ARIMA dimodelkan ulang dengan LSTM sederhana
# ─────────────────────────────────────────────

def grid_search_arima(series, p_range=range(0,4), d_range=range(0,2), q_range=range(0,4)):
    best_aic, best_order, results = np.inf, (1,1,1), []
    for p,d,q in itertools.product(p_range, d_range, q_range):
        try:
            fit = ARIMA(series, order=(p,d,q)).fit()
            results.append({"order":(p,d,q),"AIC":fit.aic,"BIC":fit.bic})
            if fit.aic < best_aic:
                best_aic, best_order = fit.aic, (p,d,q)
        except: continue
    return best_order, pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)

HYBRID_RESULTS = {}
BEST_ORDERS    = {}

print("Fitting Hybrid ARIMA-LSTM per emiten ...")
print("-" * 55)

for ticker in TICKER_LIST:
    seri   = df_monthly[ticker].dropna()

    # Step 1: ARIMA untuk komponen linear
    order, _ = grid_search_arima(seri)
    BEST_ORDERS[ticker] = order
    arima_fit = ARIMA(seri, order=order).fit()
    arima_pred = arima_fit.fittedvalues.dropna()
    arima_resid = (seri.loc[arima_pred.index] - arima_pred).values.reshape(-1, 1)

    # Step 2: LSTM untuk residual non-linear
    scaler_r  = MinMaxScaler(feature_range=(-1, 1))
    resid_scl = scaler_r.fit_transform(arima_resid)

    X_r, y_r = [], []
    for j in range(LOOK_BACK, len(resid_scl)):
        X_r.append(resid_scl[j-LOOK_BACK:j, 0])
        y_r.append(resid_scl[j, 0])
    X_r = np.array(X_r).reshape(-1, LOOK_BACK, 1)
    y_r = np.array(y_r)

    split = int(len(X_r) * 0.80)
    model_r = Sequential([
        LSTM(32, return_sequences=False, input_shape=(LOOK_BACK, 1)),
        Dropout(0.15),
        Dense(1),
    ])
    model_r.compile(optimizer=Adam(1e-3), loss="huber")
    model_r.fit(X_r[:split], y_r[:split],
                validation_data=(X_r[split:], y_r[split:]),
                epochs=100, batch_size=8,
                callbacks=[EarlyStopping(patience=12, restore_best_weights=True, verbose=0)],
                verbose=0)

    # Step 3: Forecast ARIMA 252 bulan
    fc_arima   = arima_fit.get_forecast(steps=FORECAST_STEPS)
    arima_fc_m = fc_arima.predicted_mean.values

    # Step 4: Forecast residual LSTM 252 bulan (iterative)
    window_r = resid_scl[-LOOK_BACK:].flatten().tolist()
    resid_fc = []
    for _ in range(FORECAST_STEPS):
        xr = np.array(window_r[-LOOK_BACK:]).reshape(1, LOOK_BACK, 1)
        yr = model_r.predict(xr, verbose=0)[0, 0]
        resid_fc.append(yr)
        window_r.append(yr)
    resid_fc_orig = scaler_r.inverse_transform(
        np.array(resid_fc).reshape(-1, 1)
    ).flatten()

    # Step 5: Hybrid = ARIMA forecast + LSTM residual correction
    hybrid_fc = arima_fc_m + resid_fc_orig

    last_date = seri.index[-1]
    fc_idx    = pd.date_range(
        start   = last_date + pd.offsets.MonthEnd(1),
        periods = FORECAST_STEPS,
        freq    = "ME"
    )

    HYBRID_RESULTS[ticker] = {
        "arima_fc":  arima_fc_m,
        "resid_fc":  resid_fc_orig,
        "hybrid_fc": hybrid_fc,
        "index":     fc_idx,
    }
    print(f"  {ticker:4s}  ARIMA{order} + LSTM-residual  |  "
          f"Hybrid akhir = IDR {hybrid_fc[-1]:,.0f}")

print("-" * 55)
print("[OK] Model Hybrid ARIMA-LSTM selesai.")

## 11. Visualisasi Perbandingan ARIMA vs LSTM vs Hybrid (Figure B3)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 20), sharex=False)

for ax, ticker in zip(axes, TICKER_LIST):
    seri    = df_monthly[ticker].dropna()
    w       = TICKER_COLORS[ticker]
    lstm_fc = LSTM_FORECAST[ticker]
    hyb     = HYBRID_RESULTS[ticker]

    # Historis
    ax.plot(seri.index, seri.values, color=C["gray"], lw=1.0, alpha=0.80, label="Historis")

    # ARIMA
    ax.plot(hyb["index"], hyb["arima_fc"],
            color=w, lw=1.4, ls="--", alpha=0.90, label=f"ARIMA{BEST_ORDERS[ticker]}")

    # LSTM
    ax.plot(lstm_fc["index"], lstm_fc["pred"],
            color=C["orange"], lw=1.4, ls="-.", alpha=0.90, label="LSTM")

    # Hybrid
    ax.plot(hyb["index"], hyb["hybrid_fc"],
            color=C["purple"], lw=1.8, alpha=0.95, label="Hybrid ARIMA-LSTM")

    ax.axvline(seri.index[-1], color=C["gray"], lw=0.9, ls=":", alpha=0.75)
    ax.set_ylabel(f"{ticker}\n(IDR)", fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(4))

axes[0].set_title(
    "Figure B3. Forecast Comparison: ARIMA vs LSTM vs Hybrid ARIMA-LSTM (2025–2045)",
    fontsize=12, fontweight="bold", pad=10
)
axes[-1].set_xlabel("Year", fontsize=10)
plt.tight_layout(h_pad=0.5)
fig.savefig(f"{DIR_FIG}/figB3_arima_lstm_hybrid_comparison.png", dpi=300)
plt.show()
print("[SAVED] figB3_arima_lstm_hybrid_comparison.png")

## 12. Tabel Perbandingan Evaluasi: ARIMA vs LSTM vs Hybrid

In [ ]:
# Re-evaluasi ARIMA in-sample untuk perbandingan fair
compare_rows = []

for ticker in TICKER_LIST:
    seri      = df_monthly[ticker].dropna()
    arima_fit = ARIMA(seri, order=BEST_ORDERS[ticker]).fit()
    arima_pred = arima_fit.fittedvalues.dropna()
    actual_a   = seri.loc[arima_pred.index]

    mae_a  = mean_absolute_error(actual_a, arima_pred)
    rmse_a = np.sqrt(mean_squared_error(actual_a, arima_pred))
    mape_a = np.mean(np.abs((actual_a - arima_pred) / actual_a)) * 100
    r2_a   = 1 - np.sum((actual_a - arima_pred)**2) / np.sum((actual_a - actual_a.mean())**2)

    le     = LSTM_EVAL[ticker]

    compare_rows.extend([
        {"Emiten": ticker, "Model": f"ARIMA{BEST_ORDERS[ticker]}",
         "MAE": f"{mae_a:,.2f}", "RMSE": f"{rmse_a:,.2f}", "MAPE (%)": f"{mape_a:.4f}", "R²": f"{r2_a:.6f}"},
        {"Emiten": ticker, "Model": "LSTM (2-layer)",
         "MAE": f"{le['mae']:,.2f}", "RMSE": f"{le['rmse']:,.2f}", "MAPE (%)": f"{le['mape']:.4f}", "R²": f"{le['r2']:.6f}"},
    ])

tbl_cmp = pd.DataFrame(compare_rows)
tbl_cmp.to_csv(f"{DIR_DATA}/tabelB2_perbandingan_model.csv", index=False)
print(tbl_cmp.to_string(index=False))
print("\n[SAVED] tabelB2_perbandingan_model.csv")

## 13. Ekspor Gabungan JSON (ARIMA + LSTM + Hybrid → Dashboard)

In [ ]:
output = {}

for ticker in TICKER_LIST:
    seri    = df_monthly[ticker].dropna()
    lstm_fc = LSTM_FORECAST[ticker]
    hyb     = HYBRID_RESULTS[ticker]

    historis = [
        {"date": d.strftime("%Y-%m"), "price": round(float(p), 2)}
        for d, p in zip(seri.index, seri.values)
    ]
    lstm_list = [
        {"date": d.strftime("%Y-%m"), "price": round(float(p), 2)}
        for d, p in zip(lstm_fc["index"], lstm_fc["pred"])
    ]
    hybrid_list = [
        {
            "date":    d.strftime("%Y-%m"),
            "arima":   round(float(a), 2),
            "lstm":    round(float(l), 2),
            "hybrid":  round(float(h), 2),
        }
        for d, a, l, h in zip(
            hyb["index"],
            hyb["arima_fc"],
            lstm_fc["pred"],
            hyb["hybrid_fc"],
        )
    ]

    output[ticker] = {
        "historis":          historis,
        "forecast_lstm":     lstm_list,
        "forecast_hybrid":   hybrid_list,
    }

json_path = f"{DIR_OUT}/lstm_hybrid_forecast_2025_2045.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"[SAVED] {json_path}")
for ticker in TICKER_LIST:
    print(f"  {ticker}: {len(output[ticker]['historis'])} historis  "
          f"+ {len(output[ticker]['forecast_lstm'])} LSTM forecast  "
          f"+ {len(output[ticker]['forecast_hybrid'])} Hybrid forecast")

## 14. Figure Akhir: Normalized Hybrid Forecast 4 Emiten (Figure B4)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for ticker in TICKER_LIST:
    seri  = df_monthly[ticker].dropna()
    hyb   = HYBRID_RESULTS[ticker]
    w     = TICKER_COLORS[ticker]
    basis = seri.iloc[0]

    ax.plot(seri.index,    seri / basis * 100,
            color=w, lw=0.9, alpha=0.80, label=f"{ticker} Historis")
    ax.plot(hyb["index"], hyb["hybrid_fc"] / basis * 100,
            color=w, lw=1.6, ls="--", alpha=0.95, label=f"{ticker} Hybrid")

ax.axvline(pd.Timestamp("2025-01-31"), color=C["gray"], lw=1.0, ls=":")
ax.annotate("← Historis  |  Forecast →",
            xy=(pd.Timestamp("2025-01-31"), ax.get_ylim()[1] * 0.95),
            fontsize=8, color=C["gray"], xytext=(6, 0), textcoords="offset points")

ax.set_ylabel("Indeks Harga Ternormalisasi (Basis 100, Jan-2014)", fontsize=10)
ax.set_xlabel("Year", fontsize=10)
ax.set_title(
    "Figure B4. Normalized Hybrid ARIMA-LSTM Forecast — All IDX Coal Stocks (2025–2045)",
    fontsize=12, fontweight="bold", pad=10
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
plt.tight_layout()
fig.savefig(f"{DIR_FIG}/figB4_hybrid_normalized.png", dpi=300)
plt.show()
print("[SAVED] figB4_hybrid_normalized.png")

## 15. Catatan Metodologis

| Aspek | Keterangan |
|---|---|
| **Data** | Harga penutupan bulanan 4 emiten IDX batubara: ADRO, PTBA, ITMG, BUMI |
| **Periode historis** | Jan 2014 – Des 2024 (132 bulan) |
| **Periode forecast** | Jan 2025 – Des 2045 (252 bulan) |
| **LSTM** | 2-layer (64 → 32 units) + Dropout(0.20) + Dense(16) + Dense(1) |
| **Hybrid** | ARIMA (linear trend) + LSTM (residual non-linear), iterative multi-step |
| **Look-back window** | 12 bulan |
| **Training split** | 80% train / 20% validation |
| **Loss function** | Huber loss (robust terhadap outlier) |
| **Output** | JSON dashboard · CSV evaluasi · Figure PNG 300 dpi |

### Struktur Output JSON Dashboard

```json
{
  "ADRO": {
    "historis":        [ {"date": "2014-01", "price": 1050.00}, ... ],
    "forecast_lstm":   [ {"date": "2025-01", "price": 3200.00}, ... ],
    "forecast_hybrid": [ {"date": "2025-01", "arima": 3150.0, "lstm": 3200.0, "hybrid": 3175.0}, ... ]
  },
  ...
}
```

> **Referensi lanjutan:** Hasil kedua notebook ini (ARIMA + LSTM + Hybrid) dapat digabungkan
> untuk ditampilkan di dashboard `forecastsahambatubara2045.netlify.app` melalui
> file JSON yang sudah diekspor.
